# 06 — Risk Segmentation

The final step: turn the best model's predicted probabilities into actionable
Low/Medium/High risk tiers, then check how those tiers distribute across
income, education, and healthcare-access subgroups. This is the health-equity
angle from the README — does predicted risk concentrate in groups that also
face more barriers to care?

Structure:
1. Load model and generate risk scores on the test set
2. Bucket into Low / Medium / High risk tiers
3. Risk tier distribution overall
4. Risk by income
5. Risk by education
6. Risk by healthcare access (NoDocbcCost, AnyHealthcare)
7. High-risk group summary
8. Interpretation

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import json
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import load_config, set_seed, resolve_path
from src.train import load_model
from src.segmentation import (
    bucket_risk_scores, crosstab_risk_by_subgroup,
    plot_risk_by_subgroup, summarize_high_risk_group,
)

config = load_config()
set_seed(config["random_seed"])
FIG_DIR = resolve_path(config["paths"]["figures_dir"])
MODELS_DIR = resolve_path(config["paths"]["models_dir"])

## 1. Load Model and Generate Risk Scores

Using the test set here (not train) so the risk scores reflect genuinely
held-out predictions, not predictions on data the model was fit on.

In [ ]:
with open(MODELS_DIR / "model_metadata.json") as f:
    metadata = json.load(f)
print(f"Using: {metadata['best_model_name']}")

model = load_model(str(MODELS_DIR / "best_model.joblib"))

processed_dir = resolve_path("data/processed")
X_test = pd.read_parquet(processed_dir / "X_test.parquet")
y_test = pd.read_parquet(processed_dir / "y_test.parquet").iloc[:, 0]

risk_probabilities = model.predict_proba(X_test)[:, 1]
print(f"Risk score range: {risk_probabilities.min():.3f} - {risk_probabilities.max():.3f}")

**Important caveat:** `X_test` here is the *scaled/encoded* feature set used
for modeling — subgroup columns like Income and Education are still present
but unscaled (they passed through the preprocessing pipeline unchanged per
notebook 03), so their raw category values (1-8, 1-6) remain directly
readable and usable for the crosstabs below.

## 2. Bucket into Risk Tiers

Using default thresholds (Low <0.33, Medium 0.33-0.66, High >0.66) on the
predicted *probability*, not the binary class label — this distinguishes a
borderline case (p=0.51) from a clear one (p=0.95), both of which the model
would otherwise just call 'diabetic'.

In [ ]:
risk_tier = bucket_risk_scores(risk_probabilities)
X_test_with_risk = X_test.copy()
X_test_with_risk["risk_tier"] = risk_tier.values
X_test_with_risk["actual_diabetes"] = y_test.values
X_test_with_risk[["risk_tier", "actual_diabetes"]].head()

## 3. Risk Tier Distribution Overall

In [ ]:
tier_counts = risk_tier.value_counts()
tier_pct = risk_tier.value_counts(normalize=True) * 100
print(pd.DataFrame({"count": tier_counts, "pct": tier_pct.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
tier_counts.reindex(["Low", "Medium", "High"]).plot(kind="bar", ax=ax, color=["#4c72b0", "#dd8452", "#c44e52"])
ax.set_title("Risk Tier Distribution (Test Set)")
ax.set_ylabel("Count")
plt.savefig(FIG_DIR / "19_risk_tier_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

Sanity check worth doing here: within the High tier, what fraction actually
have diabetes (per `actual_diabetes`)? This should be noticeably higher than
the base rate (~14%) if the model's risk scores are meaningful, not just
confident-sounding noise.

In [ ]:
actual_rate_by_tier = X_test_with_risk.groupby("risk_tier", observed=True)["actual_diabetes"].mean() * 100
print("Actual diabetes rate within each predicted risk tier:")
print(actual_rate_by_tier.round(1))

## 4. Risk by Income

Income is coded 1 (lowest) to 8 (highest) in this dataset. Checking whether
predicted risk concentrates at the lower end — if so, that's a population
that may also face more barriers to acting on a risk flag (see section 6).

In [ ]:
income_ct = crosstab_risk_by_subgroup(risk_tier, X_test["Income"])
plot_risk_by_subgroup(income_ct, "Risk Tier by Income Level (1=lowest, 8=highest)", save_path=str(FIG_DIR / "20_risk_by_income.png"))
plt.show()
income_ct.round(1)

## 5. Risk by Education

In [ ]:
education_ct = crosstab_risk_by_subgroup(risk_tier, X_test["Education"])
plot_risk_by_subgroup(education_ct, "Risk Tier by Education Level (1=lowest, 6=highest)", save_path=str(FIG_DIR / "21_risk_by_education.png"))
plt.show()
education_ct.round(1)

## 6. Risk by Healthcare Access

This is the sharpest health-equity question: if the people flagged highest-
risk are *also* disproportionately the ones who skip care due to cost
(`NoDocbcCost=1`) or lack any healthcare coverage (`AnyHealthcare=0`), a risk
score alone doesn't close that gap — it flags a population that may
structurally struggle to act on the flag.

In [ ]:
nodoc_ct = crosstab_risk_by_subgroup(risk_tier, X_test["NoDocbcCost"])
plot_risk_by_subgroup(nodoc_ct, "Risk Tier by 'Skipped Doctor Due to Cost' (0=No, 1=Yes)", save_path=str(FIG_DIR / "22_risk_by_nodoc_cost.png"))
plt.show()
nodoc_ct.round(1)

In [ ]:
healthcare_ct = crosstab_risk_by_subgroup(risk_tier, X_test["AnyHealthcare"])
plot_risk_by_subgroup(healthcare_ct, "Risk Tier by Healthcare Coverage (0=No, 1=Yes)", save_path=str(FIG_DIR / "23_risk_by_healthcare_coverage.png"))
plt.show()
healthcare_ct.round(1)

## 7. High-Risk Group Summary

One compact table: for the High risk tier specifically, how does its
composition compare to the overall test population across every subgroup
checked above? A subgroup level significantly overrepresented in High risk
relative to its overall share is the actionable finding to highlight in the
README.

In [ ]:
high_risk_summary = summarize_high_risk_group(
    X_test, risk_tier, subgroup_cols=["Income", "Education", "NoDocbcCost", "AnyHealthcare"],
)
high_risk_summary["overrepresentation"] = (
    high_risk_summary["pct_in_high_risk_group"] - high_risk_summary["pct_in_overall_population"]
)
high_risk_summary.sort_values("overrepresentation", ascending=False).round(2)

In [ ]:
high_risk_summary.to_csv("../reports/high_risk_group_summary.csv", index=False)
print("Saved to reports/high_risk_group_summary.csv")

## 8. Interpretation

_Fill in after reviewing the tables/plots above:_
- _Does the High risk tier's actual diabetes rate (section 3) confirm the
  model's scores are meaningful, not just confident noise?_
- _Which subgroup level is most overrepresented in the High risk tier
  (section 7) — and does it also face a healthcare-access barrier?_
- _What this implies for how a real screening program using this model
  should be designed (e.g. pairing a risk flag with low-cost follow-up
  options for lower-income flagged individuals)_

This closes out the core analysis. See the README's Future Improvements
section for next steps (deployment, fairness auditing, the 3-class target).